In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ตั้งค่าการแสดงผล DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def calculate_metrics(df, current_time=0):
    """
    ฟังก์ชันช่วยคำนวณค่าต่างๆ ตามสมการในเอกสาร
    Ci = Completion Time (เวลาเสร็จ)
    Li = Lateness (Ci - Di)
    Ti = Tardiness (max(0, Li))
    Slack = Di - CurrentTime - ti
    """
    # Flow Time / Completion Time (Ci) assumes start at 0 and cumulative sum
    df['Completion_Time (Ci)'] = df['Processing_Time (ti)'].cumsum()
    
    # Flow Time (Fi) - ในกรณีเครื่องจักรเดียวเริ่มต้นที่ 0, Fi = Ci
    df['Flow_Time (Fi)'] = df['Completion_Time (Ci)']

    # Lateness (Li) = Ci - di [cite: 37]
    if 'Due_Date (di)' in df.columns:
        df['Lateness (Li)'] = df['Completion_Time (Ci)'] - df['Due_Date (di)']
        
        # Tardiness (Ti) = max(0, Li) [cite: 43]
        df['Tardiness (Ti)'] = df['Lateness (Li)'].apply(lambda x: max(0, x))
        
        # Slack (SLi) = di - t_now - ti [cite: 134]
        df['Slack (SLi)'] = df['Due_Date (di)'] - current_time - df['Processing_Time (ti)']
        
    return df

print("Setup Complete. Ready to process scheduling algorithms.")

Setup Complete. Ready to process scheduling algorithms.


In [3]:
# --- ข้อมูลตั้งต้นจาก ตัวอย่างที่ 7.2 และ 7.3 [cite: 85, 93] ---
data = {
    'Job': [1, 2, 3, 4, 5, 6, 7, 8],
    'Processing_Time (ti)': [5, 8, 6, 3, 10, 14, 7, 3],
    'Due_Date (di)': [15, 10, 15, 25, 20, 40, 45, 50]
}
df_jobs = pd.DataFrame(data)

# ==========================================
# 2.1 SPT (Shortest Processing Time) [cite: 62]
# ==========================================
print("\n--- 2.1 SPT Rule (ตัวอย่างที่ 7.2/7.3) ---")
# เรียงลำดับตาม Processing Time น้อยไปมาก
df_spt = df_jobs.sort_values(by=['Processing_Time (ti)', 'Job']).copy()
df_spt = calculate_metrics(df_spt)

print(df_spt[['Job', 'Processing_Time (ti)', 'Completion_Time (Ci)', 'Due_Date (di)', 'Lateness (Li)']])
print(f"Mean Flow Time: {df_spt['Flow_Time (Fi)'].mean():.3f}")
print(f"Mean Lateness: {df_spt['Lateness (Li)'].mean():.3f}") # [cite: 100]

# ==========================================
# 2.2 WSPT (Weighted Shortest Processing Time) [cite: 101]
# ==========================================
print("\n--- 2.2 WSPT Rule (ตัวอย่างที่ 7.4) [cite: 109] ---")
# เพิ่มข้อมูลน้ำหนักความสำคัญ (Weight)
df_wspt = df_jobs.copy()
df_wspt['Weight (wi)'] = [1, 2, 3, 1, 2, 3, 2, 1] # ข้อมูลจากตารางที่ 7.8 [cite: 112]

# คำนวณ Ratio ti/wi
df_wspt['Ratio'] = df_wspt['Processing_Time (ti)'] / df_wspt['Weight (wi)']

# เรียงลำดับตาม Ratio น้อยไปมาก [cite: 107]
df_wspt = df_wspt.sort_values(by='Ratio').copy()
df_wspt = calculate_metrics(df_wspt)

print(df_wspt[['Job', 'Processing_Time (ti)', 'Weight (wi)', 'Ratio', 'Completion_Time (Ci)']])
weighted_mean_flow = (df_wspt['Flow_Time (Fi)'] * df_wspt['Weight (wi)']).sum() / df_wspt['Weight (wi)'].sum()
print(f"Weighted Mean Flow Time: {weighted_mean_flow:.3f}") # ควรได้ประมาณ 27.47 [cite: 118]

# ==========================================
# 2.3 EDD (Earliest Due Date) [cite: 119]
# ==========================================
# ==========================================
# 2.3 EDD (Earliest Due Date) - Updated with Formulas form Image
# ==========================================
print("\n--- 2.3 EDD Rule (ตัวอย่างที่ 7.5) ---")

# เรียงลำดับตาม Due Date น้อยไปมาก
df_edd = df_jobs.sort_values(by='Due_Date (di)').copy()
df_edd = calculate_metrics(df_edd)

# แสดงตารางผลลัพธ์
print(df_edd[['Job', 'Processing_Time (ti)', 'Due_Date (di)', 'Completion_Time (Ci)', 'Lateness (Li)', 'Tardiness (Ti)']])

# --- การคำนวณตามสมการในรูปภาพ ---

# 1. Mean Lateness (L_bar) - สมการที่ (7.5)
mean_lateness = df_edd['Lateness (Li)'].mean()

# 2. Mean Tardiness (T_bar) - สมการที่ (7.6)
mean_tardiness = df_edd['Tardiness (Ti)'].mean()

# 3. Number of Tardy Jobs (N_T) - สมการที่ (7.7)
# นับจำนวนงานที่ Ti > 0 (หรือ Li > 0)
n_tardy = (df_edd['Tardiness (Ti)'] > 0).sum()

# 4. Max Lateness (L_max) - สมการที่ (7.9)
l_max = df_edd['Lateness (Li)'].max()

# 5. Max Tardiness (T_max) - สมการที่ (7.8)
# T_max = max(0, L_max) หรือค่าสูงสุดของ Ti ก็ได้ผลเท่ากัน
t_max = max(0, l_max) 

print("\n--- สรุปผลการคำนวณ (Metrics) ---")
print(f"Mean Lateness (L_bar) [Eq 7.5]: {mean_lateness:.3f} ชั่วโมง")
print(f"Mean Tardiness (T_bar) [Eq 7.6]: {mean_tardiness:.3f} ชั่วโมง")
print(f"Number of Tardy Jobs (N_T) [Eq 7.7]: {n_tardy} งาน")
print(f"Max Lateness (L_max) [Eq 7.9]: {l_max} ชั่วโมง")
print(f"Max Tardiness (T_max) [Eq 7.8]: {t_max} ชั่วโมง")

# ==========================================
# 2.4 Slack Time [cite: 130]
# ==========================================
print("\n--- 2.4 Slack Time Rule (ตัวอย่างที่ 7.6) [cite: 136] ---")
# คำนวณ Slack ก่อนจัดเรียง (ที่ t=0)
df_slack = df_jobs.copy()
df_slack['Slack (SLi)'] = df_slack['Due_Date (di)'] - 0 - df_slack['Processing_Time (ti)']

# เรียงลำดับตาม Slack น้อยไปมาก
df_slack = df_slack.sort_values(by='Slack (SLi)').copy()
df_slack = calculate_metrics(df_slack)

print(df_slack[['Job', 'Processing_Time (ti)', 'Due_Date (di)', 'Slack (SLi)', 'Completion_Time (Ci)']])

# ==========================================
# 2.5 Hodgson's Algorithm [cite: 145]
# ==========================================
print("\n--- 2.5 Hodgson's Algorithm (ตัวอย่างที่ 7.7) [cite: 152] ---")

def hodgsons_algorithm(df_input):
    # Step 1: Order by EDD [cite: 147]
    current_schedule = df_input.sort_values(by='Due_Date (di)').to_dict('records')
    removed_jobs = []
    
    while True:
        # คำนวณ Completion Time และ Lateness ของลำดับปัจจุบัน
        current_time = 0
        first_tardy_index = -1
        
        for i, job in enumerate(current_schedule):
            current_time += job['Processing_Time (ti)']
            job['Completion_Time (Ci)'] = current_time
            job['Lateness (Li)'] = current_time - job['Due_Date (di)']
            
            # Step 2: Find first tardy job [cite: 148]
            if job['Lateness (Li)'] > 0:
                first_tardy_index = i
                break
        
        # ถ้าไม่มีงานล่าช้าเลย ให้หยุด (Step 1 condition)
        if first_tardy_index == -1:
            break
            
        # Step 3: Find job with max processing time from start to first_tardy_index [cite: 149]
        # ดูงานตั้งแต่ต้นจนถึงงานที่เสร็จไม่ทัน
        jobs_to_consider = current_schedule[:first_tardy_index+1]
        
        # หางานที่มี Processing Time มากที่สุดในกลุ่มนี้
        max_proc_job = max(jobs_to_consider, key=lambda x: x['Processing_Time (ti)'])
        
        print(f"  > พบงานล่าช้าที่ลำดับ {first_tardy_index+1} (Job {current_schedule[first_tardy_index]['Job']})")
        print(f"  > เลือกย้าย Job {max_proc_job['Job']} (Time: {max_proc_job['Processing_Time (ti)']}) ออกจากลำดับ")
        
        # ย้ายงานนั้นไปไว้ในรายการ removed
        removed_jobs.append(max_proc_job)
        current_schedule.remove(max_proc_job)
        
    # Step 4: เอาเนื้องานที่ถูกตัดออก ไปต่อท้าย [cite: 151]
    final_schedule = current_schedule + removed_jobs
    return pd.DataFrame(final_schedule)

df_hodgson = hodgsons_algorithm(df_jobs.copy())
# คำนวณ metrics รอบสุดท้าย
df_hodgson = calculate_metrics(df_hodgson)
print("\nFinal Sequence by Hodgson's:")
print(df_hodgson[['Job', 'Processing_Time (ti)', 'Due_Date (di)', 'Completion_Time (Ci)', 'Lateness (Li)']])


--- 2.1 SPT Rule (ตัวอย่างที่ 7.2/7.3) ---
   Job  Processing_Time (ti)  Completion_Time (Ci)  Due_Date (di)  Lateness (Li)
3    4                     3                     3             25            -22
7    8                     3                     6             50            -44
0    1                     5                    11             15             -4
2    3                     6                    17             15              2
6    7                     7                    24             45            -21
1    2                     8                    32             10             22
4    5                    10                    42             20             22
5    6                    14                    56             40             16
Mean Flow Time: 23.875
Mean Lateness: -3.625

--- 2.2 WSPT Rule (ตัวอย่างที่ 7.4) [cite: 109] ---
   Job  Processing_Time (ti)  Weight (wi)     Ratio  Completion_Time (Ci)
2    3                     6            3  2.000000    

In [ ]:
# ข้อมูลตัวอย่างที่ 7.8 [cite: 191]
data_parallel = {
    'Job': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Processing_Time (ti)': [5, 6, 3, 8, 7, 2, 3, 5, 4, 2],
    'Due_Date (di)': [8, 9, 14, 12, 11, 5, 8, 10, 15, 7] # เพิ่มข้อมูล Due Date จากตาราง 7.19 [cite: 271]
}
df_p = pd.DataFrame(data_parallel)
m_machines = 3 # จำนวนเครื่องจักร

def parallel_machine_scheduling(df, m, rule='SPT'):
    """
    ฟังก์ชันจัดลำดับงานบนเครื่องจักรขนาน
    rule: 'SPT', 'LPT', 'EDD'
    """
    # Step 1: Sort jobs [cite: 186, 215, 266]
    if rule == 'SPT':
        sorted_jobs = df.sort_values(by='Processing_Time (ti)').to_dict('records')
    elif rule == 'LPT':
        sorted_jobs = df.sort_values(by='Processing_Time (ti)', ascending=False).to_dict('records')
    elif rule == 'EDD':
        sorted_jobs = df.sort_values(by='Due_Date (di)').to_dict('records')
    else:
        sorted_jobs = df.to_dict('records')

    # เตรียมตัวแปรเก็บงานของแต่ละเครื่อง
    machines = [[] for _ in range(m)]
    machine_times = [0] * m # เวลาสะสมของแต่ละเครื่อง
    
    # Step 2: Assign job to machine with min load 
    for job in sorted_jobs:
        # หาเครื่องที่มีงานน้อยที่สุด
        min_machine_idx = machine_times.index(min(machine_times))
        
        # กำหนดเวลาเริ่มและเสร็จ
        start_time = machine_times[min_machine_idx]
        end_time = start_time + job['Processing_Time (ti)']
        
        # บันทึกข้อมูล
        job_record = job.copy()
        job_record['Start'] = start_time
        job_record['Finish'] = end_time
        machines[min_machine_idx].append(job_record)
        
        # อัปเดตเวลาเครื่องจักร
        machine_times[min_machine_idx] = end_time
        
    return machines, machine_times

def plot_gantt(machines, title):
    """สร้าง Gantt Chart อย่างง่าย """
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = plt.cm.tab20.colors
    
    for m_idx, jobs in enumerate(machines):
        for j_idx, job in enumerate(jobs):
            width = job['Finish'] - job['Start']
            ax.barh(y=m_idx + 1, width=width, left=job['Start'], 
                    edgecolor='black', color=colors[job['Job'] % len(colors)], align='center')
            ax.text(job['Start'] + width/2, m_idx + 1, f"J{job['Job']}", 
                    ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    
    ax.set_yticks(range(1, len(machines) + 1))
    ax.set_ylabel('Machine')
    ax.set_xlabel('Time')
    ax.set_title(title)
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    plt.show()

# --- 3.1 SPT Rule (Example 7.8) [cite: 188] ---
print("\n--- 3.1 Parallel Machine: SPT Rule ---")
machines_spt, times_spt = parallel_machine_scheduling(df_p, m_machines, 'SPT')
print(f"Makespan (SPT): {max(times_spt)}") # [cite: 212]
plot_gantt(machines_spt, "Parallel Machine Scheduling (SPT)")

# --- 3.2 LPT then SPT Rule (Example 7.9) [cite: 218] ---
# Note: ตัวอย่าง 7.9 ใช้ LPT ในการ assign เครื่อง แล้วค่อยจัด SPT ภายในเครื่องนั้นทีหลัง [cite: 217]
print("\n--- 3.2 Parallel Machine: LPT Allocation -> SPT Sequencing ---")
machines_lpt, _ = parallel_machine_scheduling(df_p, m_machines, 'LPT')

# Re-sort jobs inside each machine by SPT [cite: 243]
final_machines_lpt_spt = []
final_makespan = 0
for m_jobs in machines_lpt:
    # Sort by processing time inside the machine
    m_jobs.sort(key=lambda x: x['Processing_Time (ti)'])
    
    # Re-calculate timing
    current_t = 0
    new_m_jobs = []
    for job in m_jobs:
        job['Start'] = current_t
        job['Finish'] = current_t + job['Processing_Time (ti)']
        current_t = job['Finish']
        new_m_jobs.append(job)
    final_machines_lpt_spt.append(new_m_jobs)
    if current_t > final_makespan: final_makespan = current_t

print(f"Makespan (LPT+SPT): {final_makespan}") # [cite: 263]
plot_gantt(final_machines_lpt_spt, "Parallel Machine (LPT allocation + SPT Sequence)")

# --- 3.3 EDD Rule (Example 7.10) [cite: 268] ---
print("\n--- 3.3 Parallel Machine: EDD Rule ---")
machines_edd, times_edd = parallel_machine_scheduling(df_p, m_machines, 'EDD')
plot_gantt(machines_edd, "Parallel Machine Scheduling (EDD)")

In [ ]:
# ข้อมูลตัวอย่างที่ 7.11 [cite: 301]
data_johnson = {
    'Job': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Machine_1': [3, 6, 2, 7, 6, 5, 5, 3, 6, 10],
    'Machine_2': [5, 2, 8, 6, 6, 9, 4, 2, 8, 4]
}
df_j = pd.DataFrame(data_johnson)

print("\n--- 4. Johnson's Rule (Serial Processors) [cite: 296] ---")

def johnsons_algorithm(df):
    # เปลี่ยนเป็น list เพื่อให้จัดการง่าย
    jobs = df.to_dict('records')
    
    # ลำดับงานช่วงต้น (Front) และช่วงท้าย (Back)
    front_schedule = []
    back_schedule = []
    
    while jobs:
        # Step 1: Find global minimum processing time [cite: 297]
        # สร้าง list ของ (time, machine_id, job_index_in_list)
        min_val = float('inf')
        min_job = None
        min_machine = 0 # 1 or 2
        
        for job in jobs:
            if job['Machine_1'] < min_val:
                min_val = job['Machine_1']
                min_job = job
                min_machine = 1
            if job['Machine_2'] < min_val:
                min_val = job['Machine_2']
                min_job = job
                min_machine = 2
        
        # Step 2: Place job based on machine [cite: 297]
        if min_machine == 1:
            # ถ้าค่าน้อยสุดอยู่เครื่อง 1 ให้วางไว้หน้าสุด
            front_schedule.append(min_job)
        else:
            # ถ้าค่าน้อยสุดอยู่เครื่อง 2 ให้วางไว้ท้ายสุด
            back_schedule.insert(0, min_job)
            
        # Step 3: Remove job [cite: 298]
        jobs.remove(min_job)
        
    final_sequence = front_schedule + back_schedule
    return final_sequence

sequence_johnson = johnsons_algorithm(df_j.copy())
seq_jobs = [j['Job'] for j in sequence_johnson]
print(f"Optimal Sequence: {seq_jobs}") # [cite: 315-323]

# คำนวณ Makespan สำหรับ Johnson's Sequence
time_m1 = 0
time_m2 = 0
gantt_data = [[], []] # [Machine1_jobs, Machine2_jobs]

for job in sequence_johnson:
    # Machine 1
    start_m1 = time_m1
    finish_m1 = start_m1 + job['Machine_1']
    time_m1 = finish_m1
    
    gantt_data[0].append({
        'Job': job['Job'], 'Start': start_m1, 'Finish': finish_m1
    })
    
    # Machine 2 (เริ่มได้เมื่อ M1 เสร็จ และ M2 ว่าง)
    start_m2 = max(finish_m1, time_m2)
    finish_m2 = start_m2 + job['Machine_2']
    time_m2 = finish_m2
    
    gantt_data[1].append({
        'Job': job['Job'], 'Start': start_m2, 'Finish': finish_m2
    })

print(f"Total Makespan: {time_m2} hours") # ควรได้ 56 [cite: 324]
plot_gantt(gantt_data, "Johnson's Rule Schedule (Serial M1 -> M2)")